In [1]:
import os
import sys
sys.path.insert(0, "./cellular_cycle") # add path to the model

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import random
from pathlib import Path
import datetime
import pickle
import glob
import joblib
import umap
from statistics import mean

## Importing the model, models differ on their initial conditions, one being a concentration and the other a protein count

# to match Austin's ouput and use his functions, simulation with ODE should use omega=1 and stochastic with omega=200 (the default)
from cell_cycle_death_model import model
from model_fxns import * # functions to evaluate simulation output; includes plotObs()

##Import the simulator: for ODE and stochastic simulation :
from pysb.simulator import BngSimulator
from pysb.simulator import ScipyOdeSimulator


# Directory containing cell fate data
#DATADIR = Path('../data/Exp_Intermitotic_Times/')

# Where to save large simulation output files
SAVE_DIR = Path(Path.home(),'ECM_mod_output/dist_runs/')
#SAVE_DIR = Path(Path.home(),'ECM_mod_output')
if not SAVE_DIR.exists():
    SAVE_DIR.mkdir()

OVERWRITE = False # Overwrite existing simulation files

CURDATE = datetime.datetime.now().strftime('%Y%m%d')

### Load Data

In [8]:
file_location = os.path.join('C:\\Users\\olesk\\ECM_mod_output\\dist_runs\\', 'stoch_sim*orig.csv')
filenames = glob.glob(file_location)
filenames = sorted(filenames)
# filenames_sp = [filenames[5], filenames[6], filenames[3], filenames[4], filenames[7], filenames[8], 
#                filenames[0], filenames[1], filenames[2]]

filenames_sp = [filenames[0], filenames[3], filenames[5], filenames[8],
                filenames[11], filenames[12], filenames[18], filenames[24]]

filenames_conds = [filenames[0], filenames[12], filenames[5],
                filenames[18], filenames[11], filenames[24]]

# filenames_sp = [filenames[3], filenames[0], filenames[1], filenames[2],
#                 filenames[4], filenames[5], filenames[6], filenames[7]]
# filenames_sp=filenames

filenames_conds


['C:\\Users\\olesk\\ECM_mod_output\\dist_runs\\stoch_sim_dmso_stiff000_orig.csv',
 'C:\\Users\\olesk\\ECM_mod_output\\dist_runs\\stoch_sim_erl_stiff000_orig.csv',
 'C:\\Users\\olesk\\ECM_mod_output\\dist_runs\\stoch_sim_dmso_stiff050_orig.csv',
 'C:\\Users\\olesk\\ECM_mod_output\\dist_runs\\stoch_sim_erl_stiff050_orig.csv',
 'C:\\Users\\olesk\\ECM_mod_output\\dist_runs\\stoch_sim_dmso_stiff099_orig.csv',
 'C:\\Users\\olesk\\ECM_mod_output\\dist_runs\\stoch_sim_erl_stiff099_orig.csv']

In [9]:
file_location = os.path.join('C:\\Users\\olesk\\ECM_mod_output\\dist_runs\\', 'ODE_sim*orig.csv')
filenames_ode = glob.glob(file_location)
filenames_ode = sorted(filenames_ode)
filenames_sp_ode = [filenames_ode[0], filenames_ode[3], filenames_ode[5], filenames_ode[8],
                    filenames_ode[11], filenames_ode[12], filenames_ode[18], filenames_ode[25]]

filenames_conds_ode = [filenames_ode[0], filenames_ode[12], filenames_ode[5],
                    filenames_ode[18], filenames_ode[11], filenames_ode[25]]

# filenames_sp_ode = [filenames_ode[3], filenames_ode[0], filenames_ode[1], filenames_ode[2],
#                     filenames_ode[4], filenames_ode[5], filenames_ode[6], filenames_ode[7]]
#filenames_sp_ode = filenames_ode

filenames_conds_ode


['C:\\Users\\olesk\\ECM_mod_output\\dist_runs\\ODE_sim_dmso_stiff000_orig.csv',
 'C:\\Users\\olesk\\ECM_mod_output\\dist_runs\\ODE_sim_erl_stiff000_orig.csv',
 'C:\\Users\\olesk\\ECM_mod_output\\dist_runs\\ODE_sim_dmso_stiff050_orig.csv',
 'C:\\Users\\olesk\\ECM_mod_output\\dist_runs\\ODE_sim_erl_stiff050_orig.csv',
 'C:\\Users\\olesk\\ECM_mod_output\\dist_runs\\ODE_sim_dmso_stiff099_orig.csv',
 'C:\\Users\\olesk\\ECM_mod_output\\dist_runs\\ODE_sim_erl_stiff099_orig.csv']

In [10]:
file_location = os.path.join('C:\\Users\\olesk\\ECM_mod_output\\', 'stoch_sim*cycdi_0_4.csv')
filenames = glob.glob(file_location)
filenames = sorted(filenames)

filenames_cycdi = [filenames[2], filenames[3], filenames[0],
                 filenames[1], filenames[4], filenames[5]]

In [11]:
file_location = os.path.join('C:\\Users\\olesk\\ECM_mod_output\\', 'ode_sim*cycdi_0_4.csv')
filenames = glob.glob(file_location)
filenames = sorted(filenames)

filenames_ode_cycdi = [filenames[2], filenames[3], filenames[0],
                 filenames[1], filenames[4], filenames[5]]

In [27]:
filenames_sp

['C:\\Users\\olesk\\ECM_mod_output\\dist_runs\\stoch_sim_dmso_stiff000_orig.csv',
 'C:\\Users\\olesk\\ECM_mod_output\\dist_runs\\stoch_sim_dmso_stiff030_orig.csv',
 'C:\\Users\\olesk\\ECM_mod_output\\dist_runs\\stoch_sim_dmso_stiff050_orig.csv',
 'C:\\Users\\olesk\\ECM_mod_output\\dist_runs\\stoch_sim_dmso_stiff070_orig.csv',
 'C:\\Users\\olesk\\ECM_mod_output\\dist_runs\\stoch_sim_dmso_stiff099_orig.csv',
 'C:\\Users\\olesk\\ECM_mod_output\\dist_runs\\stoch_sim_erl_stiff000_orig.csv',
 'C:\\Users\\olesk\\ECM_mod_output\\dist_runs\\stoch_sim_erl_stiff050_orig.csv',
 'C:\\Users\\olesk\\ECM_mod_output\\dist_runs\\stoch_sim_erl_stiff099_orig.csv']

In [13]:
#The filenames to make the reducer umap object from
"""
filenames = ['erl_4-9_1_1.txt', 'erl_4-9_2_1.txt', 'erl_4-9_3_1.txt',
             'dmso_4-9_1_1.txt', 'dmso_4-9_2_1.txt', 'dmso_4-9_3_1.txt']
#Important params to set the window size and step time distance
init_dt = 0.01                #initial dt from the raw model output
dt = 0.1                      #dt for umap window
window = 4                    #window size in hours
time_step = 1                 #time step between windows

#umap params
neighbs = 100
min_d = 0.02
"""
# Important params to set the window size and step time distance
init_dt = 0.1                # initial dt from the raw model output
dt = 0.1                      # dt for umap window
window = 4                    # window size in hours
time_step = 0.5               # time step between windows

#umap params
neighbs = 50
min_d = 0.01

#reducer save name
r_name = f'reducer_03_20_{init_dt}_{dt}_{window}_{time_step}.sav'

In [14]:
def calc_minmax(filenames, filenames_ode):
    max_prots=[0]*18
    min_prots=[1000]*18
    
    for f in filenames:
        tmp_max=[]
        tmp_min=[]
        prot_tmp = pd.read_csv(f, sep=",", header=None, usecols = range(0,19))
        prot_tmp.columns = ["skp2", "p21", "pp21", "cdh1_act", "cdh1_inact", "myc", "cycD", "cycD-p21", 
        "Rb", "pRb", "E2F", "E2F-Rb", "cycE", "cycE-p21", "cycA", "cycA-p21",
        "cycB", "cycB-p21", "cdc20"]
        prot_tmp = prot_tmp.drop(columns=['cdh1_inact'])
        #prot_tmp = prot_tmp/200
        prot_tmp[prot_tmp<0] = 0 #from old extrap. method, don't think I need this
        
        tmp_max = [prot_tmp[col].max() for col in prot_tmp]
        tmp_min = [prot_tmp[col].min() for col in prot_tmp]
        
        max_prots = [max(max_prots[n], tmp_max[n]) for n in range(18)]
        min_prots = [min(min_prots[n], tmp_min[n]) for n in range(18)]
        
    for f in filenames_ode:
        tmp_max=[]
        tmp_min=[]
        prot_tmp_ode = pd.read_csv(f, sep=",", header=None, usecols = range(0,19))
        prot_tmp_ode.columns = ["skp2", "p21", "pp21", "cdh1_act", "cdh1_inact", "myc", "cycD", "cycD-p21", 
        "Rb", "pRb", "E2F", "E2F-Rb", "cycE", "cycE-p21", "cycA", "cycA-p21",
        "cycB", "cycB-p21", "cdc20"]
        prot_tmp_ode = prot_tmp_ode.drop(columns=['cdh1_inact'])
        prot_tmp[prot_tmp<0] = 0 #from old extrap. method, don't think I need this
        
        tmp_max = [prot_tmp_ode[col].max() for col in prot_tmp_ode]
        tmp_min = [prot_tmp_ode[col].min() for col in prot_tmp_ode]
        
        max_prots = [max(max_prots[n], tmp_max[n]) for n in range(18)]
        min_prots = [min(min_prots[n], tmp_min[n]) for n in range(18)]  
        
    return max_prots, min_prots

In [15]:
def load_norm_data(filenames, filenames_ode, max_prots, min_prots, time_all, init_dt, dt, window, time_step, plot_phase):
     
    if plot_phase:
        plot_phase(filenames_sp, filenames_sp_ode)
        plt.savefig(SAVE_DIR + f"phaseplot_{num}.png", bbox_inches='tight')
        plot_phase_norm(filenames_sp, filenames_sp_ode, max_prots, min_prots)
        plt.savefig(SAVE_DIR + f"phaseplot_norm_{num}.png", bbox_inches='tight')
    
    win_len = int(window/dt)
    all_prot_tmp = np.empty((0,win_len*18))     #careful, this could end up being massive 
    all_prot_tmp_ode = np.empty((0,win_len*18))
    all_time_tmp = np.empty((0,win_len))
    all_time_at_tmp = np.empty((0,1))
    range_ind = np.arange(int(win_len/2),int(len(time_tmp2)-(int(win_len/2))),int(time_step/dt))
        
    for (f, f_ode) in zip(filenames, filenames_ode):
        prot_tmp = pd.read_csv(f, sep=",", header=None, usecols = range(0,19))
        prot_tmp.columns = ["skp2", "p21", "pp21", "cdh1_act", "cdh1_inact", "myc", "cycD", "cycD-p21", 
        "Rb", "pRb", "E2F", "E2F-Rb", "cycE", "cycE-p21", "cycA", "cycA-p21",
        "cycB", "cycB-p21", "cdc20"]
        prot_tmp = prot_tmp.drop(columns=['cdh1_inact'])
        #prot_tmp = prot_tmp/200
        for ii in range(18):
            prot_tmp.iloc[:,ii] = (prot_tmp.iloc[:,ii] - min_prots[ii])/(max_prots[ii] - min_prots[ii])

        prot_re_tmp = np.empty((len(range_ind),win_len*18))
        time_re_tmp = np.empty((len(range_ind),win_len))
        time_at_tmp = np.empty((len(range_ind),1))

        ii=0
        for n in range(int(win_len/2),int(len(time_tmp2)-(int(win_len/2))),int(time_step/dt)):
            tmp = prot_tmp.iloc[n-int(win_len/2):n+int(win_len/2)]
            tmp_t = time_tmp2['time'].iloc[n-int(win_len/2):n+int(win_len/2)]
            tmp=tmp.to_numpy()
            tmp=np.reshape(np.transpose(tmp),win_len*18)
            tmp_at = time_tmp2['time'].iloc[n]

            prot_re_tmp[ii,:] = np.array([tmp])
            time_re_tmp[ii,:] = np.array([tmp_t])
            time_at_tmp[ii,:] = np.array([[tmp_at]])
            ii+=1

        all_prot_tmp = np.append(all_prot_tmp, prot_re_tmp, axis=0)
        all_time_tmp = np.append(all_time_tmp, time_re_tmp, axis=0)
        all_time_at_tmp = np.append(all_time_at_tmp, time_at_tmp, axis=0)
        
        prot_tmp_ode = pd.read_csv(f_ode, sep=",", header=None, usecols = range(0,19))
        prot_tmp_ode.columns = ["skp2", "p21", "pp21", "cdh1_act", "cdh1_inact", "myc", "cycD", "cycD-p21", 
        "Rb", "pRb", "E2F", "E2F-Rb", "cycE", "cycE-p21", "cycA", "cycA-p21",
        "cycB", "cycB-p21", "cdc20"]
        prot_tmp_ode = prot_tmp_ode.drop(columns=['cdh1_inact'])
        for ii in range(18):
            prot_tmp_ode.iloc[:,ii] = (prot_tmp_ode.iloc[:,ii] - min_prots[ii])/(max_prots[ii] - min_prots[ii])

        prot_re_tmp = np.empty((len(range_ind),win_len*18))


        ii=0
        for n in range(int(win_len/2),int(len(time_tmp2)-(int(win_len/2))),int(time_step/dt)):
            tmp = prot_tmp_ode.iloc[n-int(win_len/2):n+int(win_len/2)]
            tmp=tmp.to_numpy()
            tmp=np.reshape(np.transpose(tmp),win_len*18)

            prot_re_tmp[ii,:] = np.array([tmp])
            time_re_tmp[ii,:] = np.array([tmp_t])
            time_at_tmp[ii,:] = np.array([[tmp_at]])
            ii+=1

        all_prot_tmp_ode = np.append(all_prot_tmp_ode, prot_re_tmp, axis=0)
        
    return all_prot_tmp, all_prot_tmp_ode, all_time_tmp, all_time_at_tmp
        

In [28]:
max_prots, min_prots = calc_minmax(filenames_sp, filenames_sp_ode)

In [209]:
max_prots, min_prots = calc_minmax(filenames_conds, filenames_conds_ode)

In [29]:
time_tmp = np.linspace(0, 20000, 200001)
time_tmp2 = pd.DataFrame(data=time_tmp)
time_tmp2.columns = ["time"]
plot_phase = False

In [33]:
[all_prot_tmp, all_prot_tmp_ode, all_time_tmp, all_time_at_tmp] = load_norm_data(filenames_sp, filenames_sp_ode, max_prots, min_prots, time_tmp2, init_dt, dt, window, time_step, plot_phase)


In [30]:
[all_prot_tmp, all_prot_tmp_ode, all_time_tmp, all_time_at_tmp] = load_norm_data(filenames_conds, filenames_conds_ode, max_prots, min_prots, time_tmp2, init_dt, dt, window, time_step, plot_phase)


In [31]:
all_prot_tmp.shape

(239958, 720)

In [34]:
np.amax(all_prot_tmp)

1.0

In [154]:
###############
filenames_conds = filenames_conds + filenames_cycdi
filenames_conds_ode = filenames_conds_ode + filenames_ode_cycdi
##############

In [220]:
##############
reducer = umap.UMAP(
        n_neighbors=neighbs,
        min_dist=min_d,
        n_components=2,
        metric='euclidean',
        random_state=5,
        low_memory=True        
        ).fit(all_prot_tmp)

embedding_all = reducer.transform(all_prot_tmp)
##############

In [247]:
##########
SAVEDIR = Path(os.path.expanduser("~/ECM_mod_output/"))
reducer_filename = Path(SAVEDIR, r_name)
joblib.dump(reducer, reducer_filename)
###########

['C:\\Users\\olesk\\ECM_mod_output\\reducer_03_20_0.1_0.1_4_0.5.sav']

In [32]:
############
size = (5,5)
params = {'axes.titley': 1,
         'axes.titlepad': 5,
         'axes.labelsize': 16,
         'axes.edgecolor': "black",
         'axes.titlesize': 18,
         'xtick.labelsize': 14,
         'ytick.labelsize': 14}
plt.rcParams.update(params)

fig = plt.figure(figsize=size)
trk_len = int(len(embedding_all_prot)/len(filenames_sp))
trk_len_ode = int(len(embedding_all_prot_ode)/len(filenames_sp))
n=5
n1=6
plt.plot(embedding_all_prot[n*trk_len:n1*trk_len,0], embedding_all_prot[n*trk_len:n1*trk_len,1], 
         lw=0.05, alpha=0.7, c='gray')

# n=5
# n1=6
# plt.plot(embedding_all_prot_conds[n*trk_len:n1*trk_len,0], embedding_all_prot_conds[n*trk_len:n1*trk_len,1], 
#          lw=0.05, alpha=0.7, c='red')

# n=11
# n1=12
# plt.plot(embedding_all_prot_conds[n*trk_len:n1*trk_len,0], embedding_all_prot_conds[n*trk_len:n1*trk_len,1], 
#          lw=0.05, alpha=0.7, c='blue')

#############


NameError: name 'embedding_all_prot_ode' is not defined

<Figure size 500x500 with 0 Axes>

In [35]:
[all_prot_tmp_conds, all_prot_tmp_ode_conds, all_time_tmp, all_time_at_tmp] = load_norm_data(filenames_conds, filenames_conds_ode, max_prots, min_prots, time_tmp2, init_dt, dt, window, time_step, plot_phase)


In [36]:
[all_prot_tmp_cycdi, all_prot_tmp_ode_cycdi, all_time_tmp, all_time_at_tmp] = load_norm_data(filenames_cycdi, filenames_ode_cycdi, max_prots, min_prots, time_tmp2, init_dt, dt, window, time_step, plot_phase)


In [37]:
np.amax(all_prot_tmp_conds)

1.0

In [24]:
# reducer = joblib.load('C:\\Users\\olesk\\data\\reducer_07_25_0.1_0.1_4_0.5.sav')
reducer = joblib.load('C:\\Users\\olesk\\ECM_mod_output\\reducer_03_20_0.1_0.1_4_0.5.sav')

In [38]:
win_len = int(window/dt)

In [39]:
embedding_all_prot = reducer.transform(all_prot_tmp)

In [40]:
embedding_all_prot_conds = reducer.transform(all_prot_tmp_conds)

In [41]:
embedding_all_prot_cycdi = reducer.transform(all_prot_tmp_cycdi)

In [265]:
joblib.dump(embedding_all_prot_cycdi, "C:\\Users\\olesk\\data\\embedding_cycdi_0318.sav")
joblib.dump(all_prot_tmp_cycdi, "C:\\Users\\olesk\\data\\all_prot_cycdi_0318.sav")

['C:\\Users\\olesk\\data\\all_prot_cycdi_0318.sav']

In [266]:
joblib.dump(embedding_all_prot_conds, "C:\\Users\\olesk\\data\\embedding_0318.sav")
joblib.dump(all_prot_tmp_conds, "C:\\Users\\olesk\\data\\all_prot_0318.sav")

['C:\\Users\\olesk\\data\\all_prot_0318.sav']

In [267]:
all_prot_tmp.shape

(319944, 720)

In [42]:
win_len = int(window/dt)
all_prot_tmp_ode_ab = np.empty((0,win_len*18))
ab_len = 2000
w_len = len(all_prot_tmp_ode)/len(filenames_sp)
num_ts = int(all_prot_tmp_ode.shape[0]/w_len)

In [43]:
w_len

39993.0

In [44]:
all_prot_tmp_ode.shape

(319944, 720)

In [45]:
win_len = int(window/dt)
all_prot_tmp_ode_ab = np.empty((0,win_len*18))
ab_len = 2000
w_len = len(all_prot_tmp_ode)/len(filenames_sp)
sp = 1
for n in range(len(filenames_sp)):
    tmp = all_prot_tmp_ode[int(sp*w_len-ab_len):int(sp*w_len),:]
    all_prot_tmp_ode_ab = np.append(all_prot_tmp_ode_ab, tmp, axis=0)
    sp += 1

In [46]:
all_prot_tmp_ode_ab.shape

(16000, 720)

In [47]:
win_len = int(window/dt)
all_prot_tmp_ode_ab_conds = np.empty((0,win_len*18))
ab_len = 2000
w_len = len(all_prot_tmp_ode_conds)/len(filenames_conds)
sp = 1
for n in range(len(filenames_conds)):
    tmp = all_prot_tmp_ode_conds[int(sp*w_len-ab_len):int(sp*w_len),:]
    all_prot_tmp_ode_ab_conds = np.append(all_prot_tmp_ode_ab_conds, tmp, axis=0)
    sp += 1

In [48]:
all_prot_tmp_ode_ab_conds.shape

(12000, 720)

In [ ]:
joblib.dump(all_prot_tmp_ode_ab_conds, Path(SAVEDIR, f'all_ode_ab_{CURDATE}.sav'))